# 2. Running `vadd` on gem5

Notebook 01 stopped at an object file. Here it is linked with the runtime and run
on gem5, where an out-of-order x86 core at 2 GHz issues the commands to a DDR4
memory model that executes them with DRAM timing.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

NAME, BANKS, MEM = "vadd_1048576", 32, "ddr"
k = pud.compile_kernel(NAME, BANKS, MEM)     # steps 1-6 of notebook 01, and the link
space = pud.row_space(k.pud)
print("binary:", k.exe.relative_to(pud.ROOT))
for key, v in space.items():
    print(f"  pud.{key:20} {v}")

## The link

The kernel is linked with a small **runtime** (`pud_runtime.c`) and **driver**
(`pud_main.c`) from MIMDRAM, and the numbers above are passed to them as `-D`
defines. That is the contract between the compiler and the runtime: the compiler
decided the shape of the row space, and the runtime allocates exactly that and
publishes its start as `pud_arena`. The driver runs the kernel once cold, resets
gem5's statistics, runs it again and dumps them, so the measured run is not
charged for fetching the program.

The link uses **gcc 4.8 against Ubuntu 14.04's C library** (`pud-gcc`), the way
MIMDRAM's own microbenchmarks are built: this gem5's syscall emulation cannot run
the startup code of a current C library.

## The simulation

In [ ]:
s = pud.simulate(k.exe, MEM)
print(f"row ops   {s.row_ops}")
print(f"makespan  {s.makespan_us:.3f} us   (first DRAM command to last)")
print(f"simulated in {s.seconds:.0f} s of wall time")

## Check the count by hand

An *n*-bit addition costs `8n + 1` commands on this substrate (notebook 01, step 4),
so a 16-bit add is 129. There are 128 tiles.

In [ ]:
expected = 128 * (8 * 16 + 1)
print(f"128 x 129 = {expected}; gem5 counted {s.row_ops}:",
      "match" if expected == s.row_ops else "MISMATCH")

The number of commands the mapping priced, the number the binary issued and the
number the trace in notebook 04 expands to all agree.

## What the processor did

In [ ]:
insts, cycles = s.extra["committed_insts"], s.extra["cycles"]
print(f"committed instructions  {insts:,.0f}   ({insts / s.row_ops:.1f} per DRAM command)")
print(f"core cycles             {cycles:,.0f}   ({cycles / s.row_ops:.2f} per DRAM command)")

A few instructions per command, retired at a couple of instructions per cycle:
the core needs a few cycles to hand the memory each command. Within one channel
that is faster than the memory can execute them, so the core is not the
bottleneck. Notebooks 03 and 04 show where that changes.

The full statistics are in the `m5out_…` directory under `tutorial/artifacts/`.
Two things to know when reading them yourself: the file holds **two** dumps and
only the first is the measured run, and the row-op counters are **per memory
controller** and have to be summed.

In [ ]:
pud.show(s.outdir / "stats.txt", grep=r"rowOp(Count|FirstTick|LastTick)", lines=8)

> **Note:** the makespan can move by a few parts in ten thousand with the length of
> the binary's path, because gem5 places the program's stack after its arguments.
> The command count never moves.